In [2]:
import torch
from torch_geometric.nn import HeteroDictLinear

# Cada tipo tem uma dimensionalidade de feature diferente
lin = HeteroDictLinear(
    in_channels={'user': 64, 'item': 32, 'tag': 16},
    out_channels=128,                 # saída uniforme
)

x_dict = {
    'user': torch.randn(10, 64),
    'item': torch.randn(50, 32),
    'tag':  torch.randn(20, 16),
}

out = lin(x_dict)
print(out['user'].shape)   # torch.Size([10, 128])
print(out['item'].shape)   # torch.Size([50, 128])
print(out['tag'].shape)    # torch.Size([20, 128])

torch.Size([10, 128])
torch.Size([50, 128])
torch.Size([20, 128])


In [4]:
from pathlib import Path
import sys
import copy
import torch

root_dir = Path("/home/milnner00/projects/gnn_recsys/code")
sys.path.append(str(root_dir))

from recsys.bipartite_graph import random_80_20
from recsys.datasets import ML32MGraph

In [5]:
graph = ML32MGraph("/home/milnner00/projects/gnn_recsys/data/ml-32m")

In [6]:
hetero_ml32m = graph.to_hetero()

In [ ]:
hetero_ml32m_splits = random_80_20(graph,seed=43)
train_idx, val_idx, test_idx = hetero_ml32m_splits.idxs()
ei_t = hetero_ml32m["user", "rates", "movie"].edge_index[:,train_idx]
order1 = torch.argsort(ei_t[1],stable=True)
order2 = torch.argsort(ei_t[0,order1], stable=True)
order = order1[order2]
ei_t_s = ei_t[:,order]
src = ei_t_s[0]



nós de origem únicos: 200948
min arestas por nó (ativos): 6
max arestas por nó (ativos): 23393
média: 111.47234344482422


In [14]:
unique_users, counts = torch.unique(src, return_counts=True)
print("nós de origem únicos:", unique_users.numel())
print("min arestas por nó (ativos):", counts.min().item())
print("max arestas por nó (ativos):", counts.max().item())
print("10 >= arestas por nó (ativos):", (counts >= 10).sum().item())
print("Qnt. de arestas para 10 >= arestas por nó (ativos):", counts[(counts >= 10)].sum().item())

print("média:", counts.float().mean().item())
print("Nº arestas:", counts.sum())

nós de origem únicos: 200948
min arestas por nó (ativos): 6
max arestas por nó (ativos): 23393
10 >= arestas por nó (ativos): 200797
Qnt. de arestas para 10 >= arestas por nó (ativos): 22398843
média: 111.47234344482422
Nº arestas: tensor(22400144)


In [8]:
num_users = ei_t_s[0].max().item() + 1     # ou o número real de usuários do dataset
degrees = torch.bincount(src, minlength=num_users)

print("min grau (inclui zeros):", degrees.min().item())
print("max grau:", degrees.max().item())
print("usuários com grau 0 no treino:", (degrees == 0).sum().item())

min grau (inclui zeros): 6
max grau: 23393
usuários com grau 0 no treino: 0
